# POC HyperFrames Colab T4

Verify npx hyperframes render chay headless

In [ ]:
# Cell 1: Cài đặt toàn bộ dependencies
!apt-get update -qq
# Chromium/Puppeteer runtime deps (full list)
!apt-get install -y -qq ca-certificates fonts-liberation libappindicator3-1 libasound2 libatk-bridge2.0-0 libatk1.0-0 libc6 libcairo2 libcups2 libdbus-1-3 libexpat1 libfontconfig1 libgbm1 libgcc1 libglib2.0-0 libgtk-3-0 libnspr4 libnss3 libpango-1.0-0 libpangocairo-1.0-0 libstdc++6 libx11-6 libx11-xcb1 libxcb1 libxcomposite1 libxcursor1 libxdamage1 libxext6 libxfixes3 libxi6 libxrandr2 libxrender1 libxss1 libxtst6 lsb-release wget xdg-utils 2>&1 | tail -3
# FFmpeg
!apt-get install -y -qq ffmpeg 2>&1 | tail -2
# Node.js 22
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash - 2>&1 | tail -2
!apt-get install -y nodejs 2>&1 | tail -2
!node --version && !npm --version
print("All deps installed")


In [ ]:
import os
PROJ="/content/test_project"
os.makedirs(PROJ,exist_ok=True)
!ffmpeg -y -f lavfi -i anullsrc=r=24000:cl=mono -t 5 {PROJ}/audio.mp3 2>&1 | tail -2
s1="<!doctype html><html><head><meta charset=UTF-8></head><body style=margin:0;padding:0;background:#080B14;width:1080px;height:1920px;display:flex;align-items:center;justify-content:center;font-family:Arial><div style=text-align:center;color:white><h1 style=font-size:80px;font-weight:800;background:linear-gradient(135deg,#6366f1,#06b6d4);-webkit-background-clip:text;-webkit-text-fill-color:transparent>HYPERFRAMES POC</h1><p style=font-size:40px;color:#94a3b8>Colab T4 + Headless Chrome</p></div></body></html>"
with open(f"{PROJ}/s1.html","w") as f: f.write(s1)
idx="<!doctype html><html><head><meta charset=UTF-8><meta name=viewport content=width=1080,height=1920></head><body style=margin:0;padding:0;background:#080B14;overflow:hidden;width:1080px;height:1920px><div id=root data-composition-id=test data-width=1080 data-height=1920 data-start=0 data-duration=5><audio id=my-audio src=audio.mp3 data-start=0 data-duration=5 data-track-index=0 data-volume=1></audio><div id=s1 data-composition-src=./s1.html data-start=0 data-duration=5 data-track-index=1></div></div><script src=https://cdn.jsdelivr.net/npm/gsap@3.14.2/dist/gsap.min.js></script><script>window.__timelines=window.__timelines||{};window.__timelines.test=gsap.timeline({paused:!0});</script></body></html>"
with open(f"{PROJ}/index.html","w") as f: f.write(idx)
print("Project created")

In [ ]:
import subprocess,time,os
print("Running hyperframes...")
start=time.time()
r=subprocess.run(["npx","--yes","hyperframes@0.6.40","render","/content/test_project","--output","/content/test_project/output.mp4"],capture_output=True,text=True,timeout=300,cwd="/content/test_project")
print(f"Exit:{r.returncode} Time:{time.time()-start:.0f}s")
out="/content/test_project/output.mp4"
if r.returncode==0 and os.path.exists(out):
    print(f"SUCCESS {os.path.getsize(out)/1024:.0f}KB")
else:
    print("STDERR:",r.stderr[-500:] if r.stderr else "empty")
    print("FAILED")